In [1]:
# 🖥️ NOTEBOOK 07: FLEET MANAGEMENT DASHBOARD (ULTIMATE VERSION)
# Tính năng: Tìm Store -> Auto Xe | Legend | Phân biệt chiều đi/về

# 0. Cài đặt & Load Thư viện
!pip install polyline requests ipywidgets --quiet
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import json
import folium
from folium import plugins
import requests
import polyline
import math
import numpy as np
from google.colab import drive
import os

# 1. Load Data & Pre-processing
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics'
DATA_OUTPUT = os.path.join(BASE_PATH, 'data/03_output')

result_path = os.path.join(DATA_OUTPUT, 'optimized_routes_final.json')
with open(result_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

all_routes = data if isinstance(data, list) else data['routes']

# --- CẤU TRÚC DỮ LIỆU ---
hierarchy_data = {}
store_mapping = {} # Map: Store Name -> {Depot, Vehicle}

for r in all_routes:
    veh_id = r['vehicle_id']
    # Fix Depot ID
    depot_id = r.get('depot_id')
    if not depot_id or depot_id == 'Unknown':
        if 'D1' in veh_id: depot_id = 'DEPOT_001'
        elif 'D3' in veh_id: depot_id = 'DEPOT_003'
        else: depot_id = 'OTHER'

    # Build Hierarchy
    if depot_id not in hierarchy_data: hierarchy_data[depot_id] = {}
    if veh_id not in hierarchy_data[depot_id]: hierarchy_data[depot_id][veh_id] = []
    hierarchy_data[depot_id][veh_id].append(r)

    # Build Store Mapping
    for node in r['path']:
        if node['type'] not in ['Depot', 'Public Station']:
            store_name = node['name']
            store_mapping[store_name] = {'depot': depot_id, 'vehicle': veh_id}

# Sort Trips
for depot in hierarchy_data:
    for veh in hierarchy_data[depot]:
        hierarchy_data[depot][veh].sort(key=lambda x: x['start_time'])

# List Store cho Dropdown
store_list = sorted(list(store_mapping.keys()))

print(f"✅ Dữ liệu sẵn sàng! Quản lý {len(store_list)} điểm giao hàng.")

# 2. Hàm hỗ trợ (OSRM, Bearing)
def get_osrm_route(waypoints):
    if len(waypoints) > 60:
        step = len(waypoints) // 60 + 1
        waypoints = waypoints[::step]
        if waypoints[-1] != waypoints[-1]: waypoints.append(waypoints[-1])

    coords_str = ";".join([f"{lon},{lat}" for lat, lon in waypoints])
    url = f"http://router.project-osrm.org/route/v1/driving/{coords_str}?overview=full&geometries=polyline"
    try:
        r = requests.get(url, timeout=2)
        if r.status_code == 200:
            return polyline.decode(r.json()['routes'][0]['geometry'])
    except: pass
    return waypoints

def get_bearing(p1, p2):
    lat1, long1 = math.radians(p1[0]), math.radians(p1[1])
    lat2, long2 = math.radians(p2[0]), math.radians(p2[1])
    dLon = long2 - long1
    y = math.sin(dLon) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dLon)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def create_arrow_marker(point, angle, color, opacity):
    return folium.RegularPolygonMarker(
        location=point, fill_color=color, color=color,
        fill_opacity=opacity, opacity=opacity,
        number_of_sides=3, radius=6, rotation=angle - 90
    )

def find_nearest_index(coord, path):
    path_arr = np.array(path)
    dists = np.sum((path_arr - np.array(coord))**2, axis=1)
    return np.argmin(dists)

# 3. Hàm Thêm Mục Lục (Legend)
def add_legend(m):
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; width: 230px; height: 320px;
                border:2px solid grey; z-index:9999; font-size:13px;
                background-color:rgba(255, 255, 255, 0.9);
                padding: 10px; border-radius: 10px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
    <h4 style="margin-top:0; text-align:center; color:#2E7D32;">🗺️ CHÚ GIẢI</h4>
    <b>📍 ĐỊA ĐIỂM:</b><br>
    <i class="fa fa-warehouse" style="color:black; width:20px; text-align:center"></i> Kho Tổng (Depot)<br>
    <i class="fa fa-box" style="color:#1E88E5; width:20px; text-align:center"></i> Điểm Giao Hàng<br>
    <i class="fa fa-bolt" style="color:#43A047; width:20px; text-align:center"></i> Sạc Tranh Thủ (Mall)<br>
    <i class="fa fa-charging-station" style="color:#D32F2F; width:20px; text-align:center"></i> Sạc Chủ Động (Trạm)<br>
    <hr style="margin: 5px 0;">
    <b>🛣️ ĐƯỜNG ĐI:</b><br>
    <span style="color:#555; font-weight:bold;">─────</span> Chiều Đi (Đậm)<br>
    <span style="color:#999; font-weight:bold;">- - - -</span> Chiều Về (Mờ)<br>
    <span style="color:#555;">▶ ▶</span> Hướng di chuyển<br>
    <hr style="margin: 5px 0;">
    <b>🎨 MÀU SẮC CHUYẾN:</b><br>
    <span style="color:#D50000;">■</span> Chuyến 1 &nbsp; <span style="color:#2962FF;">■</span> Chuyến 2<br>
    <span style="color:#00C853;">■</span> Chuyến 3 &nbsp; <span style="color:#FFD600;">■</span> Chuyến 4
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

# 4. Hàm Vẽ Map Chính
# --- HÀM VẼ MAP (UPGRADE: ICONS & POPUP INFO) ---
def draw_fleet_map(depot_id, vehicle_id, selected_trip_index=None):
    if not vehicle_id or vehicle_id not in hierarchy_data.get(depot_id, {}): return None
    trips = hierarchy_data[depot_id][vehicle_id]
    start_node = trips[0]['path'][0]

    m = folium.Map(location=[start_node['lat'], start_node['long']], zoom_start=13, tiles='CartoDB positron')
    TRIP_COLORS = ['#D50000', '#2962FF', '#00C853', '#FFD600', '#AA00FF']

    for idx, trip in enumerate(trips):
        if selected_trip_index is not None and selected_trip_index != 'ALL' and idx != selected_trip_index: continue

        color = TRIP_COLORS[idx % len(TRIP_COLORS)]

        # --- Draw Line (Giữ nguyên logic cũ) ---
        waypoints = [[node['lat'], node['long']] for node in trip['path']]
        real_path = get_osrm_route(waypoints) if len(waypoints) < 60 else waypoints
        last_delivery_node = trip['path'][-2]
        split_idx = find_nearest_index([last_delivery_node['lat'], last_delivery_node['long']], real_path)

        folium.PolyLine(real_path[:split_idx+1], color=color, weight=4, opacity=0.8, tooltip=f"Trip {idx+1} (Đi)").add_to(m)
        folium.PolyLine(real_path[split_idx:], color=color, weight=4, opacity=0.3, tooltip=f"Trip {idx+1} (Về)", dash_array='5, 5').add_to(m)

        # --- Draw Arrows (Giữ nguyên logic cũ) ---
        if len(real_path) > 5:
            step = max(5, len(real_path) // 15)
            for i in range(2, len(real_path) - 2, step):
                p1, p2 = real_path[i], real_path[min(i+2, len(real_path)-1)]
                if p1 == p2: continue
                opacity = 0.8 if i <= split_idx else 0.3
                create_arrow_marker(p1, get_bearing(p1, p2), color, opacity).add_to(m)

        # --- DRAW MARKERS (LOGIC MỚI) ---
        for node in trip['path']:
            if node['type'] == 'Depot': continue

            # 1. Tính toán thông tin hiển thị
            demand = node.get('demand', 0)
            charged = node.get('charged', 0)
            soc = node.get('soc', 'N/A')

            # Thời gian giả định (để hiển thị cho đẹp)
            service_time = 15 if demand > 0 else 0
            charge_time = int(charged * 2) if charged > 0 else 0 # Giả sử sạc nhanh 2p/kWh

            # Tạo nội dung Popup chi tiết
            popup_html = f"""
            <div style='font-family: Arial; font-size: 12px; width: 180px;'>
                <b style='color:#1565C0'>{node['name']}</b><hr style='margin:5px 0'>
                🚩 <b>Trip:</b> {idx+1}<br>
                🔋 <b>Pin sau khi đến:</b> {soc}%<br>
            """

            if demand > 0:
                popup_html += f"📦 <b>Giao hàng:</b> {demand} kg<br>⏱️ <b>Thời gian dỡ:</b> {service_time}p<br>"

            if charged > 0:
                popup_html += f"⚡ <b>Đã sạc:</b> +{charged} kWh<br>⏱️ <b>Thời gian sạc:</b> {charge_time}p<br>"

            popup_html += "</div>"

            # 2. Xử lý Icon (Phân loại rõ ràng)

            # CASE A: TRẠM SẠC CHỦ ĐỘNG (Public Station) -> Icon Đỏ Duy Nhất
            if node.get('type') == 'Public Station':
                folium.Marker(
                    [node['lat'], node['long']],
                    icon=folium.Icon(color='red', icon='charging-station', prefix='fa'),
                    popup=popup_html,
                    tooltip=f"Sạc Chủ Động: {node['name']}"
                ).add_to(m)

            # CASE B: SẠC TRANH THỦ (Vừa giao hàng vừa sạc) -> Dual Icon
            elif charged > 0:
                # Icon chính: Hộp hàng (Box)
                folium.Marker(
                    [node['lat'], node['long']],
                    icon=folium.Icon(color='white', icon_color=color, icon='box', prefix='fa'),
                    popup=popup_html,
                    tooltip=f"Giao + Sạc: {node['name']}"
                ).add_to(m)

                # Icon phụ: Tia sét (Bolt) màu Xanh Lá (Green) lệch một chút
                folium.Marker(
                    [node['lat'] + 0.00015, node['long'] + 0.00015],
                    icon=folium.Icon(color='green', icon='bolt', prefix='fa'),
                    popup=f"<b style='color:green'>Sạc Tranh Thủ (+{charged} kWh)</b>",
                    zIndexOffset=500 # Nổi lên trên
                ).add_to(m)

            # CASE C: CHỈ GIAO HÀNG (Normal Delivery) -> Icon Box theo màu chuyến
            else:
                folium.Marker(
                    [node['lat'], node['long']],
                    icon=folium.Icon(color='white', icon_color=color, icon='box', prefix='fa'),
                    popup=popup_html,
                    tooltip=f"{idx+1}. {node['name']}"
                ).add_to(m)

    # Depot Marker
    folium.Marker(
        [start_node['lat'], start_node['long']],
        icon=folium.Icon(color='black', icon='warehouse', prefix='fa'),
        popup=f"<b>{depot_id}</b><br>Kho Trung Tâm",
        zIndexOffset=1000
    ).add_to(m)

    add_legend(m)
    return m

# 5. Giao diện Dashboard (4 Inputs)

dd_store = widgets.Dropdown(
    options=['-- Tìm theo Cửa Hàng --'] + store_list,
    value='-- Tìm theo Cửa Hàng --', description='🔍 Tìm:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='25%')
)

dd_depot = widgets.Dropdown(
    options=['-- Chọn Kho --'] + list(hierarchy_data.keys()),
    value='-- Chọn Kho --', description='🏭 Kho:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='25%')
)

dd_vehicle = widgets.Dropdown(
    options=[], value=None, description='🚛 Xe:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='25%'), disabled=True
)

dd_trip = widgets.Dropdown(
    options=[], value=None, description='🚩 Chuyến:',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='25%'), disabled=True
)

out_info = widgets.Output()
out_map = widgets.Output()

# --- Logic Sự Kiện ---
def on_store_change(change):
    store = change['new']
    if store != '-- Tìm theo Cửa Hàng --':
        # Auto select Depot & Vehicle
        info = store_mapping[store]
        target_depot = info['depot']
        target_veh = info['vehicle']

        # Update Depot Dropdown (No trigger)
        dd_depot.unobserve(on_depot_change, names='value')
        dd_depot.value = target_depot
        dd_depot.observe(on_depot_change, names='value')

        # Update Vehicle List & Value
        update_vehicle_list(target_depot)
        dd_vehicle.value = target_veh
    else:
        # Reset if needed
        pass

def on_depot_change(change):
    depot = change['new']
    if depot != '-- Chọn Kho --':
        update_vehicle_list(depot)
    else:
        dd_vehicle.disabled = True
        dd_trip.disabled = True

def update_vehicle_list(depot):
    vehs = hierarchy_data[depot]
    opts = []
    for v_id, t_list in vehs.items(): opts.append((f"{v_id} ({len(t_list)} chuyến)", v_id))
    dd_vehicle.options = [('-- Chọn Xe --', None)] + sorted(opts)
    dd_vehicle.value = None
    dd_vehicle.disabled = False

def on_vehicle_change(change):
    veh_id = change['new']
    depot = dd_depot.value
    if veh_id:
        trips = hierarchy_data[depot][veh_id]
        opts = [('👀 Xem TẤT CẢ', 'ALL')]
        for idx, t in enumerate(trips):
            sh, sm = divmod(t['start_time'], 60)
            eh, em = divmod(t['end_time'], 60)
            opts.append((f"Chuyến {idx+1}: {sh:02d}:{sm:02d}-{eh:02d}:{em:02d}", idx))
        dd_trip.options = opts
        dd_trip.value = 'ALL'
        dd_trip.disabled = False

        show_stats(depot, veh_id)
        with out_map:
            clear_output(wait=True)
            display(draw_fleet_map(depot, veh_id, 'ALL'))
    else:
        dd_trip.disabled = True

def on_trip_change(change):
    veh_id = dd_vehicle.value
    depot = dd_depot.value
    trip_idx = change['new']
    if veh_id and trip_idx is not None:
        with out_map:
            clear_output(wait=True)
            display(draw_fleet_map(depot, veh_id, trip_idx))

def show_stats(depot, veh_id):
    trips = hierarchy_data[depot][veh_id]
    total_km = sum(t['total_km'] for t in trips)
    total_load = sum(t['load'] for t in trips)
    total_charge = sum(sum(node.get('charged', 0) for node in t['path']) for t in trips)
    with out_info:
        clear_output()
        print(f"📊 XE: {veh_id} | Tổng: {len(trips)} Chuyến | {total_km:.1f} km | {total_load:,.0f} kg | Sạc: +{total_charge:.1f} kWh")
        for idx, t in enumerate(trips):
            sh, sm = divmod(t['start_time'], 60)
            eh, em = divmod(t['end_time'], 60)
            log = f" | ⚡ {t['logs'][0]}" if t['logs'] else ""
            print(f"   ► Trip {idx+1}: {sh:02d}:{sm:02d} -> {eh:02d}:{em:02d}{log}")

# Bind Events
dd_store.observe(on_store_change, names='value')
dd_depot.observe(on_depot_change, names='value')
dd_vehicle.observe(on_vehicle_change, names='value')
dd_trip.observe(on_trip_change, names='value')

# UI Render
ui = widgets.VBox([
    widgets.HTML("<h3 style='color:#1565C0; margin-bottom:5px'>🚀 SMART LOGISTICS CONTROL TOWER</h3>"),
    widgets.HBox([dd_store, dd_depot, dd_vehicle, dd_trip]),
    out_info,
    out_map
])

display(ui)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.8 MB/s eta 0:00:00
Mounted at /content/drive
✅ Dữ liệu sẵn sàng! Quản lý 23 điểm giao hàng.


In [4]:
# 🖥️ NOTEBOOK 07: FLEET MANAGEMENT DASHBOARD (ULTIMATE TABBED VERSION)
# Tính năng: Tìm Store -> Auto Xe | 3 Tabs (Map - Finance - Battery)

# 0. Cài đặt & Load Thư viện
!pip install polyline requests ipywidgets matplotlib seaborn --quiet
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import json
import folium
from folium import plugins
import requests
import polyline
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
import os

# 1. Load Data & Pre-processing (GIỮ NGUYÊN LOGIC CŨ)
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/datastorm_round_2_green_logistics'
DATA_OUTPUT = os.path.join(BASE_PATH, 'data/03_output')

result_path = os.path.join(DATA_OUTPUT, 'optimized_routes_final.json')
with open(result_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

all_routes = data if isinstance(data, list) else data['routes']

# --- CẤU TRÚC DỮ LIỆU ---
hierarchy_data = {}
store_mapping = {}

for r in all_routes:
    veh_id = r['vehicle_id']
    depot_id = r.get('depot_id')
    if not depot_id or depot_id == 'Unknown':
        if 'D1' in veh_id: depot_id = 'DEPOT_001'
        elif 'D3' in veh_id: depot_id = 'DEPOT_003'
        else: depot_id = 'OTHER'

    if depot_id not in hierarchy_data: hierarchy_data[depot_id] = {}
    if veh_id not in hierarchy_data[depot_id]: hierarchy_data[depot_id][veh_id] = []
    hierarchy_data[depot_id][veh_id].append(r)

    for node in r['path']:
        if node['type'] not in ['Depot', 'Public Station']:
            store_name = node['name']
            store_mapping[store_name] = {'depot': depot_id, 'vehicle': veh_id}

for depot in hierarchy_data:
    for veh in hierarchy_data[depot]:
        hierarchy_data[depot][veh].sort(key=lambda x: x['start_time'])

store_list = sorted(list(store_mapping.keys()))
print(f"✅ Dữ liệu sẵn sàng! Quản lý {len(store_list)} điểm giao hàng.")

# 2. Hàm hỗ trợ (GIỮ NGUYÊN LOGIC CŨ)
def get_osrm_route(waypoints):
    if len(waypoints) > 60:
        step = len(waypoints) // 60 + 1
        waypoints = waypoints[::step]
        if waypoints[-1] != waypoints[-1]: waypoints.append(waypoints[-1])

    coords_str = ";".join([f"{lon},{lat}" for lat, lon in waypoints])
    url = f"http://router.project-osrm.org/route/v1/driving/{coords_str}?overview=full&geometries=polyline"
    try:
        r = requests.get(url, timeout=2)
        if r.status_code == 200:
            return polyline.decode(r.json()['routes'][0]['geometry'])
    except: pass
    return waypoints

def get_bearing(p1, p2):
    lat1, long1 = math.radians(p1[0]), math.radians(p1[1])
    lat2, long2 = math.radians(p2[0]), math.radians(p2[1])
    dLon = long2 - long1
    y = math.sin(dLon) * math.cos(lat2)
    x = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dLon)
    return (math.degrees(math.atan2(y, x)) + 360) % 360

def create_arrow_marker(point, angle, color, opacity):
    return folium.RegularPolygonMarker(
        location=point, fill_color=color, color=color,
        fill_opacity=opacity, opacity=opacity,
        number_of_sides=3, radius=6, rotation=angle - 90
    )

def find_nearest_index(coord, path):
    path_arr = np.array(path)
    dists = np.sum((path_arr - np.array(coord))**2, axis=1)
    return np.argmin(dists)

def add_legend(m):
    legend_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; width: 230px; height: 320px;
                border:2px solid grey; z-index:9999; font-size:13px;
                background-color:rgba(255, 255, 255, 0.9);
                padding: 10px; border-radius: 10px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
    <h4 style="margin-top:0; text-align:center; color:#2E7D32;">🗺️ CHÚ GIẢI</h4>
    <b>📍 ĐỊA ĐIỂM:</b><br>
    <i class="fa fa-warehouse" style="color:black; width:20px; text-align:center"></i> Kho Tổng (Depot)<br>
    <i class="fa fa-box" style="color:#1E88E5; width:20px; text-align:center"></i> Điểm Giao Hàng<br>
    <i class="fa fa-bolt" style="color:#43A047; width:20px; text-align:center"></i> Sạc Tranh Thủ (Mall)<br>
    <i class="fa fa-charging-station" style="color:#D32F2F; width:20px; text-align:center"></i> Sạc Chủ Động (Trạm)<br>
    <hr style="margin: 5px 0;">
    <b>🛣️ ĐƯỜNG ĐI:</b><br>
    <span style="color:#555; font-weight:bold;">─────</span> Chiều Đi (Đậm)<br>
    <span style="color:#999; font-weight:bold;">- - - -</span> Chiều Về (Mờ)<br>
    <span style="color:#555;">▶ ▶</span> Hướng di chuyển<br>
    <hr style="margin: 5px 0;">
    <b>🎨 MÀU SẮC CHUYẾN:</b><br>
    <span style="color:#D50000;">■</span> Chuyến 1 &nbsp; <span style="color:#2962FF;">■</span> Chuyến 2<br>
    <span style="color:#00C853;">■</span> Chuyến 3 &nbsp; <span style="color:#FFD600;">■</span> Chuyến 4
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

# 4. Các hàm Vẽ (Map cũ + Finance/Battery Mới)

# --- 4A. MAP (GIỮ NGUYÊN LOGIC CŨ) ---
def draw_fleet_map(depot_id, vehicle_id, selected_trip_index=None):
    if not vehicle_id or vehicle_id not in hierarchy_data.get(depot_id, {}): return None
    trips = hierarchy_data[depot_id][vehicle_id]
    start_node = trips[0]['path'][0]

    m = folium.Map(location=[start_node['lat'], start_node['long']], zoom_start=13, tiles='CartoDB positron')
    TRIP_COLORS = ['#D50000', '#2962FF', '#00C853', '#FFD600', '#AA00FF']

    for idx, trip in enumerate(trips):
        if selected_trip_index is not None and selected_trip_index != 'ALL' and idx != selected_trip_index: continue
        color = TRIP_COLORS[idx % len(TRIP_COLORS)]
        waypoints = [[node['lat'], node['long']] for node in trip['path']]
        real_path = get_osrm_route(waypoints) if len(waypoints) < 60 else waypoints
        last_delivery_node = trip['path'][-2]
        split_idx = find_nearest_index([last_delivery_node['lat'], last_delivery_node['long']], real_path)

        folium.PolyLine(real_path[:split_idx+1], color=color, weight=4, opacity=0.8, tooltip=f"Trip {idx+1} (Đi)").add_to(m)
        folium.PolyLine(real_path[split_idx:], color=color, weight=4, opacity=0.3, tooltip=f"Trip {idx+1} (Về)", dash_array='5, 5').add_to(m)

        if len(real_path) > 5:
            step = max(5, len(real_path) // 15)
            for i in range(2, len(real_path) - 2, step):
                p1, p2 = real_path[i], real_path[min(i+2, len(real_path)-1)]
                if p1 == p2: continue
                opacity = 0.8 if i <= split_idx else 0.3
                create_arrow_marker(p1, get_bearing(p1, p2), color, opacity).add_to(m)

        for node in trip['path']:
            if node['type'] == 'Depot': continue
            demand = node.get('demand', 0)
            charged = node.get('charged', 0)
            soc = node.get('soc', 'N/A')
            service_time = 15 if demand > 0 else 0
            charge_time = int(charged * 2) if charged > 0 else 0
            popup_html = f"<div style='font-family: Arial; font-size: 12px; width: 180px;'><b style='color:#1565C0'>{node['name']}</b><hr style='margin:5px 0'>🚩 <b>Trip:</b> {idx+1}<br>🔋 <b>Pin sau khi đến:</b> {soc}%<br>"
            if demand > 0: popup_html += f"📦 <b>Giao hàng:</b> {demand} kg<br>⏱️ <b>Thời gian dỡ:</b> {service_time}p<br>"
            if charged > 0: popup_html += f"⚡ <b>Đã sạc:</b> +{charged} kWh<br>⏱️ <b>Thời gian sạc:</b> {charge_time}p<br>"
            popup_html += "</div>"

            if node.get('type') == 'Public Station':
                folium.Marker([node['lat'], node['long']], icon=folium.Icon(color='red', icon='charging-station', prefix='fa'), popup=popup_html).add_to(m)
            elif charged > 0:
                folium.Marker([node['lat'], node['long']], icon=folium.Icon(color='white', icon_color=color, icon='box', prefix='fa'), popup=popup_html).add_to(m)
                folium.Marker([node['lat'] + 0.00015, node['long'] + 0.00015], icon=folium.Icon(color='green', icon='bolt', prefix='fa'), popup=f"<b style='color:green'>Sạc Tranh Thủ (+{charged} kWh)</b>", zIndexOffset=500).add_to(m)
            else:
                folium.Marker([node['lat'], node['long']], icon=folium.Icon(color='white', icon_color=color, icon='box', prefix='fa'), popup=popup_html).add_to(m)

    folium.Marker([start_node['lat'], start_node['long']], icon=folium.Icon(color='black', icon='warehouse', prefix='fa'), popup=f"<b>{depot_id}</b><br>Kho Trung Tâm", zIndexOffset=1000).add_to(m)
    add_legend(m)
    return m

# --- 4B. FINANCE (MỚI: Tính toán chi phí cho Xe đang chọn) ---
def draw_finance_chart(depot, vehicle_id):
    trips = hierarchy_data[depot][vehicle_id]
    total_km = sum(t['total_km'] for t in trips)

    # Cost Assumptions
    cost_ev = (total_km * 0.15 * 3858) + 400000 # Điện + Khấu hao ngày
    cost_gas = (total_km * 0.09 * 24000) + 400000 # Xăng + Khấu hao ngày
    saving = cost_gas - cost_ev
    co2_saved = (total_km * 0.09 * 2.68) - (total_km * 0.15 * 0.72)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # Bar Chart
    bars = axes[0].bar(['Xe Xăng', 'Xe Điện'], [cost_gas, cost_ev], color=['#9E9E9E', '#4CAF50'])
    axes[0].set_title(f'CHI PHÍ VẬN HÀNH: {vehicle_id}', fontweight='bold')
    axes[0].grid(axis='y', linestyle='--', alpha=0.5)
    for rect in bars:
        h = rect.get_height()
        axes[0].text(rect.get_x()+rect.get_width()/2., h, f'{h:,.0f} đ', ha='center', va='bottom')

    # Pie Chart
    axes[1].pie([co2_saved, 1], labels=[f'CO2 Giảm\n({co2_saved:.1f} kg)', ''], colors=['#8BC34A', '#eee'], startangle=90, explode=(0.1, 0))
    axes[1].set_title(f'CẮT GIẢM KHÍ THẢI', fontweight='bold')

    plt.tight_layout()
    plt.show()

# --- 4C. BATTERY (MỚI: Biểu đồ Pin cho Xe đang chọn) ---
def draw_battery_chart(depot, vehicle_id, trip_idx):
    trips = hierarchy_data[depot][vehicle_id]
    target_trips = [trips[trip_idx]] if trip_idx != 'ALL' and trip_idx is not None else trips

    plt.figure(figsize=(10, 4))
    all_soc, all_labels = [], []

    for t in target_trips:
        for node in t['path']:
            all_soc.append(node.get('soc', 0))
            name = node['name']
            if len(name) > 10: name = name[:10] + "..."
            if node.get('charged', 0) > 0: name += " (⚡)"
            all_labels.append(name)

    plt.plot(range(len(all_soc)), all_soc, marker='o', linestyle='-', color='#2196F3', linewidth=2)
    plt.axhline(y=20, color='r', linestyle='--', label='Min 20%')
    plt.fill_between(range(len(all_soc)), 0, 20, color='red', alpha=0.1)

    plt.title(f"BIỂU ĐỒ PIN - {vehicle_id}", fontweight='bold')
    plt.ylabel('% Pin (SOC)')
    plt.ylim(0, 100)
    if len(all_labels) < 30:
        plt.xticks(range(len(all_labels)), all_labels, rotation=45, fontsize=8)
    else:
        plt.xticks([]) # Hide labels if too crowded
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# 5. Giao diện Dashboard (Inputs Cũ + Tabs Mới)

dd_store = widgets.Dropdown(options=['-- Tìm theo Cửa Hàng --'] + store_list, value='-- Tìm theo Cửa Hàng --', description='🔍 Tìm:', layout=widgets.Layout(width='25%'))
dd_depot = widgets.Dropdown(options=['-- Chọn Kho --'] + list(hierarchy_data.keys()), value='-- Chọn Kho --', description='🏭 Kho:', layout=widgets.Layout(width='25%'))
dd_vehicle = widgets.Dropdown(options=[], value=None, description='🚛 Xe:', layout=widgets.Layout(width='25%'), disabled=True)
dd_trip = widgets.Dropdown(options=[], value=None, description='🚩 Chuyến:', layout=widgets.Layout(width='25%'), disabled=True)

# OUTPUTS MỚI (CHIA TAB)
out_info = widgets.Output() # Giữ lại phần text stats bạn thích
out_map = widgets.Output()
out_finance = widgets.Output()
out_battery = widgets.Output()

# --- Logic Sự Kiện (Modified để update cả 3 tabs) ---
def on_store_change(change):
    store = change['new']
    if store != '-- Tìm theo Cửa Hàng --':
        info = store_mapping[store]
        dd_depot.unobserve(on_depot_change, names='value')
        dd_depot.value = info['depot']
        dd_depot.observe(on_depot_change, names='value')
        update_vehicle_list(info['depot'])
        dd_vehicle.value = info['vehicle']

def on_depot_change(change):
    depot = change['new']
    if depot != '-- Chọn Kho --': update_vehicle_list(depot)
    else:
        dd_vehicle.disabled = True
        dd_trip.disabled = True

def update_vehicle_list(depot):
    vehs = hierarchy_data[depot]
    opts = []
    for v_id, t_list in vehs.items(): opts.append((f"{v_id} ({len(t_list)} chuyến)", v_id))
    dd_vehicle.options = [('-- Chọn Xe --', None)] + sorted(opts)
    dd_vehicle.value = None
    dd_vehicle.disabled = False

def update_all_tabs(depot, veh_id, trip_idx):
    # 1. Update Info Stats
    show_stats(depot, veh_id)

    # 2. Update Map Tab
    with out_map:
        clear_output(wait=True)
        display(draw_fleet_map(depot, veh_id, trip_idx))

    # 3. Update Finance Tab
    with out_finance:
        clear_output(wait=True)
        draw_finance_chart(depot, veh_id)

    # 4. Update Battery Tab
    with out_battery:
        clear_output(wait=True)
        draw_battery_chart(depot, veh_id, trip_idx)

def on_vehicle_change(change):
    veh_id = change['new']
    depot = dd_depot.value
    if veh_id:
        trips = hierarchy_data[depot][veh_id]
        opts = [('👀 Xem TẤT CẢ', 'ALL')]
        for idx, t in enumerate(trips):
            sh, sm = divmod(t['start_time'], 60)
            eh, em = divmod(t['end_time'], 60)
            opts.append((f"Chuyến {idx+1}: {sh:02d}:{sm:02d}-{eh:02d}:{em:02d}", idx))
        dd_trip.options = opts
        dd_trip.value = 'ALL'
        dd_trip.disabled = False

        update_all_tabs(depot, veh_id, 'ALL')
    else:
        dd_trip.disabled = True

def on_trip_change(change):
    veh_id = dd_vehicle.value
    depot = dd_depot.value
    trip_idx = change['new']
    if veh_id and trip_idx is not None:
        update_all_tabs(depot, veh_id, trip_idx)

def show_stats(depot, veh_id):
    trips = hierarchy_data[depot][veh_id]
    total_km = sum(t['total_km'] for t in trips)
    total_load = sum(t['load'] for t in trips)
    total_charge = sum(sum(node.get('charged', 0) for node in t['path']) for t in trips)
    with out_info:
        clear_output()
        print(f"📊 XE: {veh_id} | Tổng: {len(trips)} Chuyến | {total_km:.1f} km | {total_load:,.0f} kg | Sạc: +{total_charge:.1f} kWh")
        for idx, t in enumerate(trips):
            sh, sm = divmod(t['start_time'], 60)
            eh, em = divmod(t['end_time'], 60)
            log = f" | ⚡ {t['logs'][0]}" if t['logs'] else ""
            print(f"   ► Trip {idx+1}: {sh:02d}:{sm:02d} -> {eh:02d}:{em:02d}{log}")

# Bind Events
dd_store.observe(on_store_change, names='value')
dd_depot.observe(on_depot_change, names='value')
dd_vehicle.observe(on_vehicle_change, names='value')
dd_trip.observe(on_trip_change, names='value')

# UI Render (Cấu trúc Tabs mới)
tabs = widgets.Tab(children=[out_map, out_finance, out_battery])
tabs.set_title(0, '🗺️ BẢN ĐỒ VẬN HÀNH')
tabs.set_title(1, '💰 HIỆU QUẢ TÀI CHÍNH')
tabs.set_title(2, '🔋 SỨC KHỎE PIN')

ui = widgets.VBox([
    widgets.HTML("<h3 style='color:#1565C0; margin-bottom:5px'>🚀 SMART LOGISTICS CONTROL TOWER (TABBED)</h3>"),
    widgets.HBox([dd_store, dd_depot, dd_vehicle, dd_trip]),
    out_info, # Phần Text Info giữ ở trên để dễ theo dõi
    tabs      # Các phần visual đưa vào Tabs
])

display(ui)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dữ liệu sẵn sàng! Quản lý 23 điểm giao hàng.
